# L9a Advanced: Stochastic Gradient Descent and CBOW Gradient Derivation
In this notebook, we derive the weight update rules for the Continuous Bag of Words (CBOW) model from first principles. We first motivate stochastic gradient descent (SGD) as an approximation to full-batch gradient descent, then derive the analytical gradients $\nabla_{\mathbf{W}_1}\mathcal{L}$ and $\nabla_{\mathbf{W}_2}\mathcal{L}$ via the chain rule. We then implement the derived update rules and verify their correctness against numerical gradients computed by finite differences.

> __Learning Objectives:__
> 
> * __Motivate stochastic gradient descent (SGD):__ Explain how SGD approximates the full-batch gradient with a single training example, and why the stochastic gradient is an unbiased estimator of the true gradient.
> * __Derive the CBOW gradient via the chain rule:__ Apply the softmax-cross-entropy gradient identity and backpropagation to derive $\nabla_{\mathbf{W}_1}\mathcal{L}$ and $\nabla_{\mathbf{W}_2}\mathcal{L}$ in closed form.
> * __Verify analytical gradients numerically:__ Implement finite-difference gradient checks to confirm the derived update rules are correct.

___

## Setup, Data, and Prerequisites

> We load packages and source files via `Include.jl`. The vocabulary and corpus are identical to those used in the CBOW example notebook so that the numerical results here are directly comparable.

In [ ]:
include("Include.jl")

### Constants

In [ ]:
sentences = [
    "I love julia and machine learning",
    "julia is fun and great",
    "I enjoy data science and machine learning",
    "data science is great and fun",
    "machine learning is a great and fun subject",
    "I love learning new things about data science"
];

control_tokens = ["<bos>", "<eos>", "<pad>", "<unk>"];

window_size = 2;
d_h         = 5;
eta         = 0.01;
num_epochs  = 500;

### Data

In [ ]:
all_words = vcat([split(lowercase(s)) for s in sentences]...)
unique_words = unique(all_words)
vocabulary = Dict{String,Int64}()
for (i, token) in enumerate(control_tokens)
    vocabulary[token] = i
end
offset = length(control_tokens)
for (i, word) in enumerate(unique_words)
    if !haskey(vocabulary, word)
        vocabulary[word] = offset + i
    end
end
inverse_vocabulary = Dict{Int64,String}(v => k for (k,v) in vocabulary)
N_V = length(vocabulary)
println("Vocabulary size: $(N_V)")

___

## Section 1: Stochastic Gradient Descent

> __Full-batch gradient descent__ minimizes the average loss over all $M$ training pairs $\{(\mathbf{x}^{(m)}, \mathbf{y}^{(m)})\}_{m=1}^{M}$:
>
> $$\mathcal{L}(\mathbf{W}_1, \mathbf{W}_2) = \frac{1}{M}\sum_{m=1}^{M} \mathcal{L}^{(m)}(\mathbf{W}_1, \mathbf{W}_2)$$
>
> Each update requires computing $\nabla_{\mathbf{W}} \mathcal{L}^{(m)}$ for every training pair before taking a single step. For a corpus with millions of sentences and a vocabulary of size $N_{\mathcal{V}} \approx 10^6$, this is computationally prohibitive.

Stochastic gradient descent replaces the full sum with a single randomly selected term.

> __Definition (Stochastic Gradient Descent):__
>
> Let $\mathcal{L}(\theta) = \frac{1}{M}\sum_{m=1}^{M}\mathcal{L}^{(m)}(\theta)$ be the average loss over $M$ training examples, where $\theta$ collects all parameters. At each iteration $t$, SGD draws one index $m_t$ uniformly at random from $\{1,\dots,M\}$ and updates:
>
> $$\theta^{(t+1)} = \theta^{(t)} - \eta\,\nabla_\theta\mathcal{L}^{(m_t)}\!\left(\theta^{(t)}\right)$$
>
> where $\eta > 0$ is the learning rate. The key property is that the stochastic gradient is an __unbiased estimator__ of the full gradient:
>
> $$\mathbb{E}_{m_t}\!\left[\nabla_\theta\mathcal{L}^{(m_t)}(\theta)\right] = \nabla_\theta\mathcal{L}(\theta)$$
>
> so each update moves in the correct direction on average, even though individual steps are noisy.

### Why noise can help

SGD introduces variance into the gradient estimate. While this makes individual steps noisier than full-batch gradient descent, the noise has a beneficial effect: it helps the optimizer escape shallow local minima and saddle points that would trap a deterministic gradient descent trajectory. This property becomes increasingly important as the loss surface grows more complex with model size.

> __Mini-batch SGD__ balances variance reduction against computational cost. Instead of a single example, a random subset $\mathcal{B}_t \subset \{1,\dots,M\}$ of size $B$ is drawn at each step:
>
> $$\theta^{(t+1)} = \theta^{(t)} - \frac{\eta}{B}\sum_{m\in\mathcal{B}_t}\nabla_\theta\mathcal{L}^{(m)}\!\left(\theta^{(t)}\right)$$
>
> The variance of the gradient estimate scales as $1/B$, so larger batches give smoother updates at the cost of more computation per step. In practice, batch sizes of 32–512 are common. Pure SGD ($B=1$) and full-batch gradient descent ($B=M$) are the two extremes.

### Comparison with gradient descent variants

Students have seen full-batch gradient descent and momentum. The table below places SGD in context:

| Method | Gradient estimate | Update rule |
|--------|-------------------|-------------|
| Full-batch GD | Exact: $\frac{1}{M}\sum_m \nabla\mathcal{L}^{(m)}$ | $\theta \leftarrow \theta - \eta\,\nabla\mathcal{L}$ |
| Momentum | Exact gradient + history | $v \leftarrow \beta v - \eta\,\nabla\mathcal{L}$; $\theta \leftarrow \theta + v$ |
| SGD (this notebook) | Stochastic: $\nabla\mathcal{L}^{(m_t)}$ | $\theta \leftarrow \theta - \eta\,\nabla\mathcal{L}^{(m_t)}$ |
| Mini-batch SGD | $\frac{1}{B}\sum_{m\in\mathcal{B}}\nabla\mathcal{L}^{(m)}$ | $\theta \leftarrow \theta - \frac{\eta}{B}\sum_{m\in\mathcal{B}}\nabla\mathcal{L}^{(m)}$ |

For CBOW, each training pair $(\mathbf{x}^{(m)}, \mathbf{y}^{(m)})$ corresponds to one (context sum, target one-hot) pair. The standard training loop iterates over all pairs in random order — this is equivalent to SGD with $B=1$ and one full pass over the data per epoch.

___

## Section 2: Deriving the CBOW Gradients

> We derive $\nabla_{\mathbf{W}_1}\mathcal{L}$ and $\nabla_{\mathbf{W}_2}\mathcal{L}$ for a single training pair $(\mathbf{x}, \mathbf{y})$. Recall the CBOW forward pass:
>
> $$\mathbf{h} = \mathbf{W}_1\,\mathbf{x}, \qquad \mathbf{u} = \mathbf{W}_2\,\mathbf{h}, \qquad \hat{y}_i = \frac{e^{u_i}}{\sum_{j=1}^{N_{\mathcal{V}}} e^{u_j}}, \qquad \mathcal{L} = -\sum_{i=1}^{N_{\mathcal{V}}} y_i\,\log\hat{y}_i$$
>
> We apply the chain rule in four steps, working backwards from $\mathcal{L}$ to $\mathbf{W}_1$.

### Step 1: Gradient of $\mathcal{L}$ with respect to $\mathbf{u}$

> We compute $\partial\mathcal{L}/\partial u_i$ using the chain rule through the softmax:
>
> $$\frac{\partial\mathcal{L}}{\partial u_i} = -\sum_{j=1}^{N_{\mathcal{V}}} y_j \frac{\partial \log \hat{y}_j}{\partial u_i} = -\sum_{j=1}^{N_{\mathcal{V}}} \frac{y_j}{\hat{y}_j}\,\frac{\partial \hat{y}_j}{\partial u_i}$$
>
> The softmax Jacobian has two cases. For $j = i$:
>
> $$\frac{\partial \hat{y}_i}{\partial u_i} = \hat{y}_i(1 - \hat{y}_i)$$
>
> For $j \neq i$:
>
> $$\frac{\partial \hat{y}_j}{\partial u_i} = -\hat{y}_j\,\hat{y}_i$$
>
> Substituting both cases:
>
> $$\frac{\partial\mathcal{L}}{\partial u_i} = -\frac{y_i}{\hat{y}_i}\,\hat{y}_i(1-\hat{y}_i) \;-\; \sum_{j \neq i} \frac{y_j}{\hat{y}_j}\left(-\hat{y}_j\,\hat{y}_i\right)$$
>
> $$= -y_i(1-\hat{y}_i) + \hat{y}_i\sum_{j \neq i} y_j$$
>
> $$= -y_i + y_i\hat{y}_i + \hat{y}_i\sum_{j \neq i} y_j = -y_i + \hat{y}_i\!\left(y_i + \sum_{j \neq i} y_j\right) = -y_i + \hat{y}_i\underbrace{\sum_{j=1}^{N_{\mathcal{V}}} y_j}_{=\,1}$$
>
> Since $\mathbf{y}$ is a one-hot vector, $\sum_j y_j = 1$, and the result simplifies to:
>
> $$\boxed{\frac{\partial\mathcal{L}}{\partial u_i} = \hat{y}_i - y_i}$$
>
> In vector form, defining $\boldsymbol{\delta} = \hat{\mathbf{y}} - \mathbf{y} \in \mathbb{R}^{N_{\mathcal{V}}}$:
>
> $$\nabla_{\mathbf{u}}\mathcal{L} = \boldsymbol{\delta} = \hat{\mathbf{y}} - \mathbf{y}$$
>
> This is the __softmax-cross-entropy identity__: the gradient equals the prediction error. The cleanliness of this result is why the softmax-cross-entropy pairing is so widely used.

### Step 2: Gradient with respect to $\mathbf{W}_2$

> Since $u_i = \sum_k W_{2,ik}\,h_k$, we have $\partial u_i / \partial W_{2,ij} = h_j$. By the chain rule:
>
> $$\frac{\partial\mathcal{L}}{\partial W_{2,ij}} = \frac{\partial\mathcal{L}}{\partial u_i}\,\frac{\partial u_i}{\partial W_{2,ij}} = \delta_i\,h_j$$
>
> Assembling all entries into matrix form gives the outer product:
>
> $$\nabla_{\mathbf{W}_2}\mathcal{L} = \boldsymbol{\delta}\,\mathbf{h}^{\top} \in \mathbb{R}^{N_{\mathcal{V}} \times d_h}$$

### Step 3: Gradient with respect to $\mathbf{h}$

> Since $u_i = \sum_k W_{2,ik}\,h_k$, we have $\partial u_i / \partial h_k = W_{2,ik}$. Summing over all output units:
>
> $$\frac{\partial\mathcal{L}}{\partial h_k} = \sum_{i=1}^{N_{\mathcal{V}}} \frac{\partial\mathcal{L}}{\partial u_i}\,\frac{\partial u_i}{\partial h_k} = \sum_{i=1}^{N_{\mathcal{V}}} \delta_i\,W_{2,ik}$$
>
> In matrix form:
>
> $$\frac{\partial\mathcal{L}}{\partial \mathbf{h}} = \mathbf{W}_2^{\top}\,\boldsymbol{\delta} \in \mathbb{R}^{d_h}$$

### Step 4: Gradient with respect to $\mathbf{W}_1$

> Since $h_k = \sum_l W_{1,kl}\,x_l$, we have $\partial h_k / \partial W_{1,kl} = x_l$ (and $\partial h_{k'}/\partial W_{1,kl} = 0$ for $k' \neq k$). By the chain rule:
>
> $$\frac{\partial\mathcal{L}}{\partial W_{1,kl}} = \frac{\partial\mathcal{L}}{\partial h_k}\,\frac{\partial h_k}{\partial W_{1,kl}} = \left(\mathbf{W}_2^{\top}\boldsymbol{\delta}\right)_k x_l$$
>
> Assembling into matrix form:
>
> $$\nabla_{\mathbf{W}_1}\mathcal{L} = \left(\mathbf{W}_2^{\top}\,\boldsymbol{\delta}\right)\mathbf{x}^{\top} \in \mathbb{R}^{d_h \times N_{\mathcal{V}}}$$

### Summary of the update rules

> __Definition (CBOW SGD Update Rules):__
>
> Let $\boldsymbol{\delta} = \hat{\mathbf{y}} - \mathbf{y} \in \mathbb{R}^{N_{\mathcal{V}}}$ be the prediction error for a single training pair $(\mathbf{x}, \mathbf{y})$, and let $\mathbf{h} = \mathbf{W}_1\mathbf{x}$. The SGD updates are:
>
> $$\mathbf{W}_2 \leftarrow \mathbf{W}_2 - \eta\,\boldsymbol{\delta}\,\mathbf{h}^{\top}$$
>
> $$\mathbf{W}_1 \leftarrow \mathbf{W}_1 - \eta\,\left(\mathbf{W}_2^{\top}\boldsymbol{\delta}\right)\mathbf{x}^{\top}$$
>
> Both updates depend only on the prediction error $\boldsymbol{\delta}$, the hidden state $\mathbf{h}$, and the input $\mathbf{x}$ — all quantities already computed during the forward pass. Note that $\mathbf{W}_2$ must be evaluated at its current value (before the update) when computing $\nabla_{\mathbf{W}_1}\mathcal{L}$.

___

## Section 3: Implementation and Numerical Verification

> We implement the derived gradients as Julia functions and verify them against numerical gradients computed by centered finite differences:
>
> $$\frac{\partial\mathcal{L}}{\partial\theta_{ij}} \approx \frac{\mathcal{L}(\theta + \epsilon\,\mathbf{E}_{ij}) - \mathcal{L}(\theta - \epsilon\,\mathbf{E}_{ij})}{2\epsilon}$$
>
> where $\mathbf{E}_{ij}$ is the matrix with a 1 in position $(i,j)$ and zeros elsewhere, and $\epsilon > 0$ is a small perturbation. If the analytical and numerical gradients agree to high relative precision, the derivation is correct.

In [ ]:
# Forward pass and loss
function cbow_loss(W1::Matrix{Float64}, W2::Matrix{Float64},
                   x::Vector{Float64}, y::Vector{Float64})::Float64
    h    = W1 * x
    u    = W2 * h
    eᵤ   = exp.(u .- maximum(u))   # numerically stable softmax
    ŷ    = eᵤ ./ sum(eᵤ)
    return -sum(y .* log.(ŷ .+ 1e-15))
end

# Analytical gradients (derived above)
function cbow_analytical_gradients(W1::Matrix{Float64}, W2::Matrix{Float64},
                                   x::Vector{Float64}, y::Vector{Float64})
    h    = W1 * x
    u    = W2 * h
    eᵤ   = exp.(u .- maximum(u))
    ŷ    = eᵤ ./ sum(eᵤ)
    δ    = ŷ .- y                  # prediction error: N_V
    dW2  = δ * h'                  # outer product: N_V × d_h
    dW1  = (W2' * δ) * x'         # outer product: d_h × N_V
    return dW1, dW2
end

We implement centered finite differences for numerical gradient verification.

In [ ]:
function cbow_numerical_gradients(W1::Matrix{Float64}, W2::Matrix{Float64},
                                   x::Vector{Float64}, y::Vector{Float64};
                                   ε::Float64 = 1e-5)
    dW1_num = zeros(size(W1))
    dW2_num = zeros(size(W2))

    for i in 1:size(W1, 1), j in 1:size(W1, 2)
        W1p = copy(W1); W1p[i,j] += ε
        W1m = copy(W1); W1m[i,j] -= ε
        dW1_num[i,j] = (cbow_loss(W1p, W2, x, y) - cbow_loss(W1m, W2, x, y)) / (2ε)
    end

    for i in 1:size(W2, 1), j in 1:size(W2, 2)
        W2p = copy(W2); W2p[i,j] += ε
        W2m = copy(W2); W2m[i,j] -= ε
        dW2_num[i,j] = (cbow_loss(W1, W2p, x, y) - cbow_loss(W1, W2m, x, y)) / (2ε)
    end

    return dW1_num, dW2_num
end

We build one training pair from the corpus, initialize weights randomly, and compare the two gradient estimates.

In [ ]:
Random.seed!(42)
training_pairs = build_cbow_pairs(sentences, vocabulary; window_size = window_size)
x_test, y_test = training_pairs[1]

W1_test = randn(d_h, N_V) * 0.01
W2_test = randn(N_V, d_h) * 0.01

dW1_an, dW2_an   = cbow_analytical_gradients(W1_test, W2_test, x_test, y_test)
dW1_num, dW2_num = cbow_numerical_gradients(W1_test, W2_test, x_test, y_test)

We measure agreement using the maximum relative error:
$$\text{rel. error} = \max_{i,j}\frac{|A_{ij} - N_{ij}|}{|A_{ij}| + |N_{ij}| + \varepsilon}$$
where $A$ is the analytical gradient and $N$ is the numerical gradient. Values below $10^{-4}$ indicate correct implementation.

In [ ]:
rel_error(A, B; ε=1e-8) = maximum(abs.(A .- B) ./ (abs.(A) .+ abs.(B) .+ ε))

err_W1 = rel_error(dW1_an, dW1_num)
err_W2 = rel_error(dW2_an, dW2_num)

println("Max relative error  ∇W₁ : $(round(err_W1, sigdigits=4))")
println("Max relative error  ∇W₂ : $(round(err_W2, sigdigits=4))")

Finally, we verify that training with the derived update rules produces the same loss trajectory as the original `train_cbow` implementation.

In [ ]:
function train_cbow_derived(training_pairs, vocab_size::Int64;
                             d_h::Int64=5, eta::Float64=0.01, num_epochs::Int64=500)
    W1 = randn(d_h, vocab_size) * 0.01
    W2 = randn(vocab_size, d_h) * 0.01
    loss_history = zeros(Float64, num_epochs)
    for epoch in 1:num_epochs
        total_loss = 0.0
        for (x, y) in training_pairs
            # forward pass
            h   = W1 * x
            u   = W2 * h
            eᵤ  = exp.(u .- maximum(u))
            ŷ   = eᵤ ./ sum(eᵤ)
            total_loss += -sum(y .* log.(ŷ .+ 1e-15))
            # backward pass (derived update rules)
            δ   = ŷ .- y
            dW2 = δ * h'
            dW1 = (W2' * δ) * x'
            W2 .-= eta .* dW2
            W1 .-= eta .* dW1
        end
        loss_history[epoch] = total_loss / length(training_pairs)
    end
    return W1, W2, loss_history
end

Random.seed!(42)
_, _, loss_derived = train_cbow_derived(training_pairs, N_V;
    d_h=d_h, eta=eta, num_epochs=num_epochs)

Random.seed!(42)
_, _, loss_original = train_cbow(training_pairs, N_V;
    d_h=d_h, eta=eta, num_epochs=num_epochs)

plot(loss_original; label="Original implementation", lw=2)
plot!(loss_derived; label="Derived update rules", lw=2, linestyle=:dash)
xlabel!("Epoch")
ylabel!("Average Loss")
title!("Loss Comparison: Original vs. Derived Update Rules")

___

## Summary

In this notebook, we derived the CBOW weight update rules from first principles. We showed that SGD approximates the full-batch gradient with a single training example, then applied the chain rule to obtain closed-form expressions for $\nabla_{\mathbf{W}_1}\mathcal{L}$ and $\nabla_{\mathbf{W}_2}\mathcal{L}$. The derivation confirmed that the analytical gradients match numerical finite-difference estimates, and that training with the derived rules reproduces the original implementation.

> __Key Takeaways__
> * **SGD is an unbiased gradient estimator:** The stochastic gradient $\nabla_\theta\mathcal{L}^{(m)}$ from a single randomly selected training pair satisfies $\mathbb{E}[\nabla_\theta\mathcal{L}^{(m)}] = \nabla_\theta\mathcal{L}$, so SGD converges to the same solution as full-batch gradient descent, but with far less computation per update.
> * **The softmax-cross-entropy gradient simplifies to the prediction error:** Differentiating the cross-entropy loss through the softmax yields $\partial\mathcal{L}/\partial u_i = \hat{y}_i - y_i$, a result that depends on the softmax Jacobian telescoping with the cross-entropy derivative because $\mathbf{y}$ is a one-hot vector.
> * **Both update rules reduce to outer products:** $\nabla_{\mathbf{W}_2}\mathcal{L} = \boldsymbol{\delta}\,\mathbf{h}^{\top}$ and $\nabla_{\mathbf{W}_1}\mathcal{L} = (\mathbf{W}_2^{\top}\boldsymbol{\delta})\,\mathbf{x}^{\top}$ are both rank-1 outer products, which means each SGD step modifies the weight matrices along a single direction determined by the current prediction error.